# Fine-Tuning LoRA no Google Colab

Fluxo alinhado à disciplina **Fine-tuning e RAG para documentos** (FIAP Fase 3):

1. **Google Colab com GPU** (Runtime → Change runtime type → T4 GPU)
2. **Google Drive** — persistir adapter e métricas (sessão Colab é temporária)
3. **Hugging Face** — token com acesso a [Llama 2 7B](https://huggingface.co/meta-llama/Llama-2-7b-hf)
4. **LoRA + PEFT + quantização 4-bit** (`train.py` do projeto)

**Antes de rodar:** solicite acesso ao modelo no Hugging Face e adicione `HF_TOKEN` em *Secrets* (ícone de chave na barra lateral).

In [ ]:
# 1) Dependências (como na aula: transformers, peft, bitsandbytes, datasets)
!pip install -q torch transformers peft bitsandbytes accelerate datasets python-dotenv huggingface_hub

## Google Drive

Monte o Drive para salvar o adapter LoRA e as métricas — sem isso, os arquivos são perdidos ao fechar o Colab.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/hospital-fiap-assistant"
DRIVE_ARTIFACTS_DIR = f"{DRIVE_PROJECT_DIR}/artifacts"
print("Drive montado. Artefatos serão copiados para:", DRIVE_ARTIFACTS_DIR)

## Repositório

Opção A: clone do GitHub (ajuste `REPO_URL` se necessário).
Opção B: upload manual da pasta do projeto em `/content/hospital-fiap-assistant`.

In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = ""  # ex.: "https://github.com/seu-usuario/hospital-fiap-assistant.git"
ROOT = Path("/content/hospital-fiap-assistant")

if not ROOT.exists() and REPO_URL:
    !git clone {REPO_URL} {ROOT}

if not ROOT.exists():
    ROOT = Path("..").resolve() if (Path("..") / "fine_tuning" / "train.py").exists() else Path(".").resolve()

os.chdir(ROOT)
print("Working directory:", ROOT)

## Hugging Face login

Adicione `HF_TOKEN` em **Secrets** do Colab ou use `login()` interativo abaixo.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF_TOKEN configurado via Colab Secrets.")
except Exception:
    print("HF_TOKEN não encontrado em Secrets — usando login interativo.")
    login(add_to_git_credential=False)
    os.environ["HF_TOKEN"] = os.environ.get("HUGGING_FACE_HUB_TOKEN", "")

In [ ]:
# Verificar GPU (obrigatório para treino real)
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU não detectada. Em Colab: Runtime → Change runtime type → GPU (T4)."
    )
print("CUDA OK:", torch.cuda.get_device_name(0))

## Dataset

Gera `data/processed/*.jsonl` se ainda não existir (PubMedQA sintético + protocolos internos).

In [ ]:
train_path = ROOT / "data/processed/train.jsonl"
if not train_path.exists():
    !python fine_tuning/prepare_dataset.py --pubmedqa data/synthetic/pubmedqa_sample.jsonl
else:
    print("Dataset já existe:", train_path)

## Treino LoRA

- `SMOKE_TEST = True` → 2 steps (validação rápida da GPU)
- `SMOKE_TEST = False` → **3 épocas completas** (recomendado para entrega)

In [ ]:
from fine_tuning.train import build_arg_parser, run_training

SMOKE_TEST = False  # True = smoke test (2 steps); False = treino completo (3 épocas)

parser = build_arg_parser()
train_args = [] if not SMOKE_TEST else ["--max-steps", "2"]
args = parser.parse_args(train_args)

print("Modo:", "smoke test (2 steps)" if SMOKE_TEST else "treino completo (3 épocas)")
metrics = run_training(args)
metrics

## Salvar no Google Drive e baixar

Copia `artifacts/lora_adapter/` e `training_metrics.json` para o Drive e gera um ZIP para download local.

In [ ]:
adapter_dir = ROOT / "artifacts" / "lora_adapter"
metrics_file = ROOT / "artifacts" / "training_metrics.json"
drive_artifacts = Path(DRIVE_ARTIFACTS_DIR)
drive_artifacts.mkdir(parents=True, exist_ok=True)

if metrics.get("skipped_training"):
    print("Treino foi ignorado (sem GPU ou erro). Nada para copiar.")
else:
    if adapter_dir.exists():
        dest_adapter = drive_artifacts / "lora_adapter"
        if dest_adapter.exists():
            shutil.rmtree(dest_adapter)
        shutil.copytree(adapter_dir, dest_adapter)
        print("Adapter copiado para:", dest_adapter)
    if metrics_file.exists():
        shutil.copy2(metrics_file, drive_artifacts / "training_metrics.json")
        print("Métricas copiadas para:", drive_artifacts / "training_metrics.json")

print("skipped_training:", metrics.get("skipped_training"))
print("output:", metrics.get("output_dir", "artifacts/lora_adapter"))
print("epochs:", metrics.get("epochs"))

In [ ]:
import zipfile

zip_path = ROOT / "artifacts" / "lora_adapter_colab.zip"
if adapter_dir.exists() and not metrics.get("skipped_training"):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path in adapter_dir.rglob("*"):
            if file_path.is_file():
                zf.write(file_path, file_path.relative_to(adapter_dir.parent))
    print("ZIP criado:", zip_path)
    print("Baixe este arquivo e extraia em artifacts/lora_adapter/ no projeto local.")
    print("Defina LORA_ADAPTER_PATH=./artifacts/lora_adapter no .env")

## Inferência rápida (opcional)

Teste o adapter treinado com uma amostra do conjunto de teste.

In [ ]:
import json

if metrics.get("skipped_training"):
    print("Pule inferência — treino não executado.")
else:
    os.environ.pop("USE_MOCK_LLM", None)
    os.environ["LORA_ADAPTER_PATH"] = str(adapter_dir)

    from assistant.llm_loader import load_llm_for_inference

    test_path = ROOT / "data/processed/test.jsonl"
    sample = json.loads(test_path.read_text(encoding="utf-8").splitlines()[0])
    prompt = sample.get("text") or sample.get("instruction", "")

    pipe = load_llm_for_inference()
    out = pipe(prompt, max_new_tokens=128, do_sample=False)
    text = out[0]["generated_text"] if isinstance(out, list) else str(out)
    print("Prompt (truncado):", prompt[:200], "...")
    print("\nResposta:", text[:500])